In [ ]:
!pip install pymupdf
!pip install nltk
!pip install tqdm
!pip install sentence-transformers
!pip install scikit-learn
!install gradio

import gradio as gr
import fitz  # PyMuPDF
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# -------------------
# PDF Parsing & Chunking
# -------------------
def extract_text_from_pdf(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text.strip()

def regex_sent_tokenize(text):
    sentence_endings = re.compile(r'(?<=[.!?])\s+(?=[A-Z])')
    return sentence_endings.split(text)

def chunk_text(text, max_tokens=100):
    sentences = regex_sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        token_count = len(sentence.split())
        if current_length + token_count > max_tokens:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_length = token_count
        else:
            current_chunk.append(sentence)
            current_length += token_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# -------------------
# Embedding & Retrieval
# -------------------
embedder = SentenceTransformer("all-MiniLM-L6-v2")
generator = pipeline("text2text-generation", model="google/flan-t5-large", device=0)

SIMILARITY_THRESHOLD = 0.3

def embed_chunks(chunks):
    embeddings = embedder.encode(chunks, show_progress_bar=False)
    return [{"text": chunk, "embedding": emb} for chunk, emb in zip(chunks, embeddings)]

def embed_query(query):
    return embedder.encode([query])[0]

def retrieve_top_k_chunks(query, chunk_data, top_k=3):
    query_vec = embed_query(query)
    chunk_vectors = np.array([item["embedding"] for item in chunk_data])
    scores = cosine_similarity([query_vec], chunk_vectors)[0]
    top_indices = scores.argsort()[::-1][:top_k]
    top_chunks = [chunk_data[i]["text"] for i in top_indices]
    avg_score = np.mean([scores[i] for i in top_indices])
    return top_chunks, avg_score

def generate_answer(query, retrieved_chunks, use_context=True):
    if use_context:
        context = "\n".join(retrieved_chunks)
        prompt = (
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\n"
            f"Please answer the question in a very detailed way based on the context."
        )
    else:
        prompt = (
            f"Question: {query}\n\n"
            f"Please provide a detailed answer."
        )

    result = generator(prompt, max_length=512, do_sample=False)[0]["generated_text"]
    return result.strip()

# -------------------
# Gradio Interface
# -------------------
document_chunks = []
document_embeddings = []

def load_document(file):
    global document_chunks, document_embeddings
    text = extract_text_from_pdf(file.name)
    chunks = chunk_text(text)
    document_embeddings = embed_chunks(chunks)
    document_chunks = chunks
    return f"Document loaded with {len(chunks)} chunks."

def answer_question(query, mode):
    if not document_embeddings and mode != "General Knowledge":
        return "Please upload a document first."

    if mode == "General Knowledge":
        return generate_answer(query, [], use_context=False)

    elif mode == "Document Only":
        top_chunks, _ = retrieve_top_k_chunks(query, document_embeddings)
        return generate_answer(query, top_chunks, use_context=True)

    elif mode == "Auto":
        top_chunks, similarity_score = retrieve_top_k_chunks(query, document_embeddings)
        use_context = similarity_score >= SIMILARITY_THRESHOLD
        return generate_answer(query, top_chunks, use_context=use_context)

    else:
        return "Invalid mode selected."

with gr.Blocks() as demo:
    gr.Markdown("## FLAN-T5 PDF Chatbot with Hybrid QA")
    gr.Markdown("Upload a PDF and ask questions either based on its content or general knowledge.")

    with gr.Row():
        pdf_input = gr.File(label="Upload PDF", type="filepath")
        upload_output = gr.Textbox(label="Status", interactive=False)

    with gr.Row():
        question_input = gr.Textbox(label="Ask a question...")
        mode_selector = gr.Radio(
            choices=["Auto", "Document Only", "General Knowledge"],
            value="Auto",
            label="Answer Mode"
        )

    with gr.Row():
        answer_button = gr.Button("Get Answer")
        answer_output = gr.Textbox(label="Answer", interactive=False, lines=6)

    pdf_input.change(fn=load_document, inputs=pdf_input, outputs=upload_output)
    answer_button.click(fn=answer_question, inputs=[question_input, mode_selector], outputs=answer_output)

demo.launch()


'install' is not recognized as an internal or external command,
operable program or batch file.
Device set to use cpu


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
